# RigTech · runner_colab

Este notebook **não contém lógica**. Ele apenas prepara o ambiente, traz os dados do Drive e chama scripts em `src/`.

## Fluxo

1. Clona repo, instala deps
2. Monta Drive, copia dataset bruto (`DaninhasTreinoClientes/`) para disco local
3. Roda pipeline: convert → split (golden) → snapshot v1 → train
4. Rsync dos resultados (pequenos: history.json, weights, plots) de volta ao Drive

## Pré-requisito no Drive

A pasta compartilhada `DaninhasTreinoClientes/` precisa aparecer no seu **Meu Drive** como atalho, dentro de `MyDrive/rigtech-weed-cycle/`. Em 'Compartilhados comigo' → botão direito na pasta → **Organizar → Adicionar atalho ao Drive** → escolher `rigtech-weed-cycle/`.

Runtime → Change runtime type → **GPU A100** (T4 se A100 indisponível — dobra o tempo do treino).

## 1. Repo e deps

In [ ]:
!git clone https://github.com/Rigtech-Solutions/rigtech-weed-cycle.git
%cd rigtech-weed-cycle

In [ ]:
!pip install -q ultralytics==8.4.115 rasterio shapely pyproj pillow pyyaml

## 2. Drive → disco local

Nunca ler do Drive montado durante processamento — a latência mata. Sempre copiar para `/content/` primeiro.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Path do atalho da pasta compartilhada dentro do teu MyDrive
SRC = '/content/drive/MyDrive/rigtech-weed-cycle/DaninhasTreinoClientes'
!ls -la "{SRC}" && echo '---' && du -sh "{SRC}"

In [ ]:
!mkdir -p work/geosource/_talhoes
!cp -r "{SRC}"/. work/geosource/_talhoes/
!ls -la work/geosource/_talhoes/

### 2b. Inspecionar layout de cada talhão

O conversor aceita 2 layouts (flat `{tif,geojson}` OU `{imagem/, daninhas/}`). Só imprimindo pra confirmar.

In [ ]:
!for d in work/geosource/_talhoes/*/; do echo "=== $d ==="; ls "$d"; done

## 3. Fase 1: converter para YOLO-seg

In [ ]:
!python -m src.convert_to_yoloseg

## 4. Fase 2: QA estático

In [ ]:
!python -m src.qa_static

## 5. Fase 3: golden set (hold-out de Flaviano01)

Se `DaninhasTreinoClientes/` tiver DoisRiosFlaviano como talhão a mais, revisar essa escolha antes de continuar.

In [ ]:
!python -m src.split_golden_by_talhao

## 6. Snapshot v1

In [ ]:
!python -m src.snapshot --note 'v1 colab: dataset completo, Flaviano01 como golden'

## 7. Treino baseline (A100, batch=16, imgsz=1024, amp)

In [ ]:
!python -m src.train_eval --version v1 --tag baseline_colab \
    --device 0 --batch 16 --imgsz 1024 --amp --epochs 100

## 8. Resultado

In [ ]:
!cat work/runs/history.json

In [ ]:
import glob
run_dirs = sorted(glob.glob('work/runs/v1_baseline_colab_*'))
run_dir = run_dirs[-1] if run_dirs else None
print('run:', run_dir)
!ls -la {run_dir}/train/ 2>/dev/null | head -30

## 9. Devolver artefatos leves ao Drive

Só o essencial (não o dataset materializado). `history.json`, `weights/best.pt`, `results.csv`, `results.png`, `confusion_matrix.png`.

**Não sobe** `work/materialized/` (dataset extraído, ~500 MB) nem `work/versions/v1/dataset.tar.gz` (437 MB) — Drive cheio.

In [ ]:
!mkdir -p /content/drive/MyDrive/rigtech-weed-cycle/runs
!cp work/runs/history.json /content/drive/MyDrive/rigtech-weed-cycle/runs/
!cp work/runs/qa_static.csv /content/drive/MyDrive/rigtech-weed-cycle/runs/
!rsync -av --include='*.pt' --include='*.png' --include='*.csv' --include='*.yaml' \
  --include='*/' --exclude='*' work/runs/ /content/drive/MyDrive/rigtech-weed-cycle/runs/